In [ ]:
    ############    #############   OpenAPI as a Machine-Readable Contract   #############   ##############   

 =>  FastAPI generates a full OpenAPI (formerly 'Swagger') schema automatically from your
       routes' type hints and Pydantic models -- available at /openapi.json, with an
       interactive UI at /docs.

 =>  This schema IS the contract between your API and every client (a frontend, another
       team's service, an external partner) -- it's precise enough that tools can generate
       client SDKs, mock servers, and automated tests directly from it.


<img src="images/contract-testing-pipeline.png" alt="API contract testing pipeline: FastAPI app to OpenAPI spec to contract tests to CI gate to deploy">

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI(title="Orders API", version="1.0.0")

class Order(BaseModel):
    id: int
    total_cents: int

@app.get("/orders/{order_id}", response_model=Order)
def get_order(order_id: int) -> Order:
    return Order(id=order_id, total_cents=1999)

client = TestClient(app)
schema = client.get("/openapi.json").json()
print("title:", schema["info"]["title"], "version:", schema["info"]["version"])
print("paths:", list(schema["paths"].keys()))
print("Order schema fields:", list(schema["components"]["schemas"]["Order"]["properties"].keys()))


In [ ]:
 =>  Nothing extra was written to produce this -- the OpenAPI schema (paths, the Order
       model's exact fields and types) came entirely from the route signature and the
       Pydantic model already needed for validation.

 =>  This is what a contract-testing tool (schemathesis, Dredd) consumes: it reads
       /openapi.json and can automatically generate test requests that check your live API
       actually matches what it claims to expose.


In [ ]:
# A minimal illustration of what 'contract testing' checks: does a response actually
# match its declared schema? (schemathesis/Dredd automate this against openapi.json --
# this is the same idea, done by hand for one field.)

response = client.get("/orders/1")
body = response.json()
order_schema = schema["components"]["schemas"]["Order"]
required_fields = set(order_schema["required"])
actual_fields = set(body.keys())

missing = required_fields - actual_fields
print("missing required fields:", missing if missing else "none -- contract satisfied")


In [ ]:
    ############    #############   Making This a CI Gate   #############   ##############   

 =>  In CI: (1) start the app, (2) fetch /openapi.json, (3) run schemathesis (or similar)
       against the live app using that schema, (4) fail the build on any mismatch -- a
       removed field, a changed type, an endpoint that no longer matches its own
       documented contract.

 =>  This catches the exact failure mode from the API Versioning notebook: someone renames
       a field on an existing (non-versioned) route and breaks every client silently -- the
       contract test fails the build BEFORE that ships.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Install schemathesis and run 'schemathesis run http://localhost:8000/openapi.json'
           against a real running instance of one of your FastAPI apps from this topic.

 =>  [ ] Add a GitHub Actions step that fails the build if the contract test step fails.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Hand-maintaining a separate API documentation page instead of relying on the
       auto-generated OpenAPI schema -- the two drift apart and the docs quietly become
       wrong.

 =>  Treating OpenAPI generation as 'nice to have docs' rather than as an enforceable
       contract -- without an actual contract test gate in CI, nothing stops a breaking
       change from shipping.
